In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

CATALOG = "workspace"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:
treatments_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.treatments")

appointments_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.appointments")

In [0]:
display(treatments_df)

treatment_id,appointment_id,treatment_type,description,cost,treatment_date,ingestion_timestamp,source_file,load_date
T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02
T002,A002,MRI,Advanced protocol,4158.44,2023-06-09,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02
T003,A003,MRI,Standard procedure,3731.55,2023-06-28,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02
T004,A004,MRI,Basic screening,4799.86,2023-09-01,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02
T005,A005,ECG,Standard procedure,582.05,2023-07-06,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02
T006,A006,Chemotherapy,Standard procedure,1381.0,2023-06-19,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02
T007,A007,Chemotherapy,Advanced protocol,534.03,2023-04-09,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02
T008,A008,Physiotherapy,Basic screening,3413.64,2023-05-24,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02
T009,A009,Physiotherapy,Standard procedure,4541.14,2023-03-05,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02
T010,A010,Physiotherapy,Standard procedure,1595.67,2023-01-13,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02


In [0]:
treatments_df.printSchema()

root
 |-- treatment_id: string (nullable = true)
 |-- appointment_id: string (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- treatment_date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_date: date (nullable = true)



In [0]:
treatments_df = treatments_df.dropDuplicates(["treatment_id"])

In [0]:
treatments_df = (

    treatments_df

    .withColumn(
        "treatment_type",
        initcap(trim(col("treatment_type")))
    )

    .withColumn(
        "description",
        initcap(trim(col("description")))
    )

)

In [0]:
treatments_df = treatments_df.join(

    appointments_df.select("appointment_id"),

    on="appointment_id",

    how="inner"

)

In [0]:
treatments_df = treatments_df.filter(
    col("cost") > 0
)

In [0]:
treatments_df = (

    treatments_df

    .withColumn(

        "cost_category",

        when(col("cost") < 1000, "Low")

        .when(col("cost") < 3000, "Medium")

        .otherwise("High")

    )

)

In [0]:
treatments_df = treatments_df.withColumn(

    "silver_load_timestamp",

    current_timestamp()

)

In [0]:
(
    treatments_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.treatments")
)

In [0]:
display(
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.treatments")
)

appointment_id,treatment_id,treatment_type,description,cost,treatment_date,ingestion_timestamp,source_file,load_date,cost_category,silver_load_timestamp
A007,T007,Chemotherapy,Advanced Protocol,534.03,2023-04-09,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,Low,2026-07-02T07:23:22.005Z
A014,T014,Ecg,Basic Screening,2082.3,2023-05-25,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,Medium,2026-07-02T07:23:22.005Z
A053,T053,Chemotherapy,Standard Procedure,1565.92,2023-02-12,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,Medium,2026-07-02T07:23:22.005Z
A085,T085,Ecg,Advanced Protocol,968.49,2023-02-18,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,Low,2026-07-02T07:23:22.005Z
A087,T087,Ecg,Advanced Protocol,3102.74,2023-10-19,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,High,2026-07-02T07:23:22.005Z
A116,T116,X-ray,Advanced Protocol,1288.86,2023-07-07,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,Medium,2026-07-02T07:23:22.005Z
A124,T124,Chemotherapy,Standard Procedure,3492.1,2023-03-16,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,High,2026-07-02T07:23:22.005Z
A139,T139,Mri,Basic Screening,4217.3,2023-10-10,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,High,2026-07-02T07:23:22.005Z
A159,T159,Chemotherapy,Advanced Protocol,4687.68,2023-04-08,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,High,2026-07-02T07:23:22.005Z
A163,T163,X-ray,Basic Screening,4450.88,2023-06-27,2026-07-02T06:36:50.119Z,treatments.csv,2026-07-02,High,2026-07-02T07:23:22.005Z


In [0]:
spark.sql("""
SELECT cost_category, COUNT(*) AS total
FROM workspace.silver.treatments
GROUP BY cost_category
ORDER BY total DESC
""").show()

+-------------+-----+
|cost_category|total|
+-------------+-----+
|       Medium|   89|
|         High|   88|
|          Low|   23|
+-------------+-----+



In [0]:
spark.sql("""
SELECT
ROUND(AVG(cost),2) AS avg_cost,
MIN(cost) AS min_cost,
MAX(cost) AS max_cost
FROM workspace.silver.treatments
""").show()

+--------+--------+--------+
|avg_cost|min_cost|max_cost|
+--------+--------+--------+
| 2756.25|  534.03| 4973.63|
+--------+--------+--------+

